# Research 73 — RealMLP-TD on the revised Task 2 dataset

Uses the attached CSVs at `/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/02_multiclass_classification`. The first cell installs `pytabkit==1.8.0` explicitly from PyPI, so notebook Internet access must be enabled. Exact SHA-256 and schema checks stop the run before fitting if the files are stale or mismatched.

On Kaggle's dual-T4 machine, independent OOF folds are trained in separate `loky` worker processes, one per GPU; the final wave runs the last OOF fold on GPU 0 while the all-train Test-prediction refit runs on GPU 1. CPU threads are divided across active fits. With fewer GPUs, it falls back to sequential folds. Model recipe and fold seed remain unchanged.

Test is prediction-only. OOF and Test outputs are keyed by `player_id`; old-snapshot OOF predictions must not be blended with these outputs. `submit/index.csv` is untouched. RealMLP-TD is trained from scratch; no pretrained weights or hyperparameter search.

In [ ]:
%pip install -q --index-url https://pypi.org/simple pytabkit==1.8.0

In [ ]:
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from joblib import Parallel, delayed, parallel_config
from pytabkit import RealMLP_TD_Classifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold

DATA_DIR = Path(
    "/kaggle/input/competitions/player-intelligence-system-cpe-342-ml/cpe342-kaggle-main/02_multiclass_classification"
)
OUTPUT_DIR = Path("/kaggle/working")
REVISION = "2026-11-12"
EXPECTED_HASHES = {
    "train.csv": "0cdaa5fad6c17a7f27d965b988cd72f6113a1a53c1f88bcf5d12c25128249f48",
    "test.csv": "224345827c690d4383fa1c81d4a7baecfb22735ed80189b1d90504dfcfa9b703",
}


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


TRAIN_PATH, TEST_PATH = DATA_DIR / "train.csv", DATA_DIR / "test.csv"
assert TRAIN_PATH.is_file() and TEST_PATH.is_file(), (
    f"Attach the competition input containing train.csv and test.csv at {DATA_DIR}."
)
actual_hashes = {"train.csv": sha256(TRAIN_PATH), "test.csv": sha256(TEST_PATH)}
assert actual_hashes == EXPECTED_HASHES, (
    f"Wrong data revision; expected {EXPECTED_HASHES}, got {actual_hashes}. Stopping before any model fit."
)
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
assert train.shape == (104_000, 46), train.shape
assert test.shape == (25_889, 45), test.shape
assert "segment" in train and "segment" not in test
assert list(train.drop(columns="segment").columns) == list(test.columns)
assert train["player_id"].notna().all() and train["player_id"].is_unique
assert test["player_id"].notna().all() and test["player_id"].is_unique
assert set(train["player_id"].astype(str)).isdisjoint(test["player_id"].astype(str))
y = train["segment"].to_numpy(dtype=int)
assert np.array_equal(np.unique(y), np.arange(4))
assert torch.cuda.is_available(), "Kaggle GPU is not enabled/available."
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print(
    {
        "revision": REVISION,
        "data_dir": str(DATA_DIR),
        "data_hashes": actual_hashes,
        "train_shape": train.shape,
        "test_shape": test.shape,
        "class_counts": train["segment"].value_counts().sort_index().to_dict(),
        "gpus": gpu_names,
        "torch": torch.__version__,
    }
)

In [ ]:
def features(frame):
    x = frame.drop(
        columns=["segment", "player_id", "random_metric_1", "random_metric_2", "random_metric_3"],
        errors="ignore",
    ).copy()
    ratios = {
        "spend_per_hour": ("total_spending_thb", "total_playtime_hours"),
        "spend_per_account_day": ("total_spending_thb", "account_age_days"),
        "playtime_per_account_day": ("total_playtime_hours", "account_age_days"),
        "monthly_spend_per_purchase": ("avg_monthly_spending", "spending_frequency"),
        "gifts_per_friend": ("gifts_sent_received", "friend_count"),
        "invites_per_friend": ("friend_invites_sent", "friend_count"),
        "ranked_per_team": ("ranked_participation_rate", "team_play_percentage"),
    }
    for name, (numerator, denominator) in ratios.items():
        x[name] = x[numerator] / (x[denominator].abs() + 1)
    x["weekly_minutes"] = x["play_frequency"] * x["avg_session_duration"]
    x["social_activity"] = x["friend_count"] * x["team_play_percentage"]
    x["missing_count"] = x.isna().sum(axis=1)
    return x.replace([np.inf, -np.inf], np.nan)


X = features(train)
X_test = features(test)
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = [column for column in X.columns if column not in cat_cols]
assert list(X.columns) == list(X_test.columns)
folds = list(StratifiedKFold(n_splits=3, shuffle=True, random_state=42).split(X, y))
print(
    f"Features={X.shape[1]} (categorical={len(cat_cols)}, numeric={len(num_cols)}); folds=3, seed=42"
)

In [ ]:
def prepare_pair(x_fit, x_pred):
    x_fit = x_fit.copy()
    x_pred = x_pred.copy()
    if cat_cols:
        x_fit[cat_cols] = x_fit[cat_cols].fillna("__MISSING__").astype(str)
        x_pred[cat_cols] = x_pred[cat_cols].fillna("__MISSING__").astype(str)
    if num_cols:
        medians = x_fit[num_cols].median().fillna(0)
        x_fit[num_cols] = x_fit[num_cols].fillna(medians)
        x_pred[num_cols] = x_pred[num_cols].fillna(medians)
    return x_fit, x_pred


def aligned_proba(model, x):
    probabilities = np.asarray(model.predict_proba(x), dtype=np.float64)
    classes = np.asarray(getattr(model, "classes_", np.arange(4)), dtype=int)
    assert probabilities.shape == (len(x), len(classes)), probabilities.shape
    assert set(classes.tolist()) == {0, 1, 2, 3}, classes
    probabilities = probabilities[:, np.argsort(classes)]
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    return probabilities


def fit_oof_fold(fold_number, fit_idx, valid_idx, device, n_threads):
    x_fit, x_valid = prepare_pair(X.iloc[fit_idx], X.iloc[valid_idx])
    started = time.perf_counter()
    model = RealMLP_TD_Classifier(device=device, random_state=42, n_threads=n_threads, verbosity=0)
    model.fit(x_fit, y[fit_idx], cat_col_names=cat_cols)
    probabilities = aligned_proba(model, x_valid)
    predictions = probabilities.argmax(axis=1)
    return {
        "fold": fold_number,
        "valid_idx": valid_idx,
        "probabilities": probabilities,
        "macro_f1": f1_score(y[valid_idx], predictions, average="macro"),
        "weighted_f1": f1_score(y[valid_idx], predictions, average="weighted"),
        "elapsed_seconds": time.perf_counter() - started,
    }


def fit_full_model(device, n_threads):
    x_full, x_test = prepare_pair(X, X_test)
    started = time.perf_counter()
    model = RealMLP_TD_Classifier(device=device, random_state=42, n_threads=n_threads, verbosity=0)
    model.fit(x_full, y, cat_col_names=cat_cols)
    probabilities = aligned_proba(model, x_test)
    return {
        "test_proba": probabilities,
        "elapsed_seconds": time.perf_counter() - started,
    }


def run_task(task):
    kind, args = task
    result = fit_oof_fold(*args) if kind == "oof" else fit_full_model(*args)
    return kind, result


parallel_gpu_count = min(torch.cuda.device_count(), 2, len(folds))
assert parallel_gpu_count >= 1, "At least one CUDA GPU is required."
cpu_thread_count = os.cpu_count() or 4
threads_per_fit = max(1, cpu_thread_count // parallel_gpu_count)
oof_proba = np.zeros((len(train), 4), dtype=np.float64)
oof_fold = np.full(len(train), -1, dtype=np.int8)
fold_rows = []
full_fit_result = None
combined_fit_started = time.perf_counter()
for wave_start in range(0, len(folds), parallel_gpu_count):
    wave = list(
        enumerate(folds[wave_start : wave_start + parallel_gpu_count], start=wave_start + 1)
    )
    tasks = [
        (
            "oof",
            (fold_number, fit_idx, valid_idx, f"cuda:{gpu_slot}", threads_per_fit),
        )
        for gpu_slot, (fold_number, (fit_idx, valid_idx)) in enumerate(wave)
    ]
    is_last_wave = wave_start + len(wave) == len(folds)
    if is_last_wave and len(wave) < parallel_gpu_count:
        tasks.append(("full", (f"cuda:{len(wave)}", threads_per_fit)))
    if len(tasks) == 1:
        completed = [run_task(tasks[0])]
    else:
        with parallel_config(
            backend="loky", n_jobs=len(tasks), inner_max_num_threads=threads_per_fit
        ):
            completed = Parallel(verbose=10)(delayed(run_task)(task) for task in tasks)
    for kind, result in completed:
        if kind == "oof":
            valid_idx = result["valid_idx"]
            oof_proba[valid_idx] = result["probabilities"]
            oof_fold[valid_idx] = result["fold"]
            fold_rows.append(
                {
                    key: value
                    for key, value in result.items()
                    if key != "probabilities" and key != "valid_idx"
                }
            )
            print(
                f"Fold {result['fold']}/3: macro F1={result['macro_f1']:.6f}; "
                f"weighted F1={result['weighted_f1']:.6f}; "
                f"fit={result['elapsed_seconds'] / 60:.1f} min",
                flush=True,
            )
        else:
            full_fit_result = result
if full_fit_result is None:
    full_fit_result = fit_full_model("cuda:0", cpu_thread_count)
test_proba = full_fit_result["test_proba"]
full_elapsed = full_fit_result["elapsed_seconds"]
fold_rows.sort(key=lambda row: row["fold"])
assert (oof_fold > 0).all()
assert np.isfinite(oof_proba).all() and np.allclose(oof_proba.sum(axis=1), 1.0, atol=1e-6)
assert np.isfinite(test_proba).all() and np.allclose(test_proba.sum(axis=1), 1.0, atol=1e-6)
combined_fit_wall_seconds = time.perf_counter() - combined_fit_started
oof_fit_seconds_sum = sum(row["elapsed_seconds"] for row in fold_rows)
print(
    f"OOF model-fit time summed across folds: {oof_fit_seconds_sum / 60:.1f} min; "
    f"OOF plus full-refit wall time: {combined_fit_wall_seconds / 60:.1f} min; "
    f"GPUs={parallel_gpu_count}, CPU threads per active fit={threads_per_fit}."
)

In [ ]:
oof_pred = oof_proba.argmax(axis=1)
macro_f1 = f1_score(y, oof_pred, average="macro")
weighted_f1 = f1_score(y, oof_pred, average="weighted")
class_report = classification_report(y, oof_pred, output_dict=True, zero_division=0)
metrics = {
    "revision": REVISION,
    "train_sha256": actual_hashes["train.csv"],
    "train_rows": int(len(train)),
    "folds": 3,
    "cv_seed": 42,
    "model": "pytabkit.RealMLP_TD_Classifier",
    "model_random_state": 42,
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "oof_fit_seconds_sum": round(oof_fit_seconds_sum, 2),
    "combined_oof_and_refit_wall_seconds": round(combined_fit_wall_seconds, 2),
    "full_fit_and_test_predict_seconds": round(full_elapsed, 2),
    "parallel_gpu_count": parallel_gpu_count,
    "threads_per_active_fit": threads_per_fit,
    "fold_results": fold_rows,
    "class_report": class_report,
    "confusion_matrix": confusion_matrix(y, oof_pred, labels=[0, 1, 2, 3]).tolist(),
    "test_used_for_training_or_validation": False,
}
score_tag = f"macro-f1-{macro_f1:.6f}"
oof_path = OUTPUT_DIR / f"segment-realmlp-td-revision-{REVISION}-oof-{score_tag}.csv"
oof_frame = pd.DataFrame(
    {
        "player_id": train["player_id"].astype(str),
        "fold": oof_fold,
        "segment": y,
        **{f"prob_{i}": oof_proba[:, i] for i in range(4)},
        "predicted_segment": oof_pred,
    }
)
assert oof_frame["player_id"].equals(train["player_id"].astype(str))
oof_frame.to_csv(oof_path, index=False)
(OUTPUT_DIR / f"segment-realmlp-td-revision-{REVISION}-oof-{score_tag}-metrics.json").write_text(
    json.dumps(metrics, indent=2), encoding="utf-8"
)
pd.DataFrame(
    confusion_matrix(y, oof_pred, labels=[0, 1, 2, 3]),
    index=[f"true_{i}" for i in range(4)],
    columns=[f"pred_{i}" for i in range(4)],
).to_csv(
    OUTPUT_DIR / f"segment-realmlp-td-revision-{REVISION}-oof-{score_tag}-confusion-matrix.csv"
)
display(
    pd.DataFrame(
        [
            {
                "candidate": "RealMLP-TD updated data, 3-fold OOF",
                "macro_f1": macro_f1,
                "weighted_f1": weighted_f1,
            }
        ]
    )
)
display(pd.DataFrame(fold_rows))
print(f"Saved OOF: {oof_path}")

In [ ]:
# Full-data training/test inference ran in the OOF cell, parallel to the final fold when a second GPU exists.
test_pred = test_proba.argmax(axis=1)
test_path = OUTPUT_DIR / f"segment-realmlp-td-revision-{REVISION}-test-{score_tag}.csv"
test_frame = pd.DataFrame(
    {
        "player_id": test["player_id"].astype(str),
        **{f"prob_{i}": test_proba[:, i] for i in range(4)},
        "segment": test_pred,
    }
)
assert test_frame["player_id"].equals(test["player_id"].astype(str))
assert len(test_frame) == len(test) and np.isfinite(test_proba).all()
assert np.allclose(test_proba.sum(axis=1), 1.0, atol=1e-6)
test_frame.to_csv(test_path, index=False)
summary = {
    **metrics,
    "test_sha256": actual_hashes["test.csv"],
    "test_rows": int(len(test)),
    "full_fit_and_test_predict_seconds": round(full_elapsed, 2),
    "test_predicted_class_share": {
        str(k): float(v)
        for k, v in pd.Series(test_pred).value_counts(normalize=True).sort_index().items()
    },
    "oof_file": oof_path.name,
    "test_prediction_file": test_path.name,
    "submission_file_modified": False,
}
summary_path = OUTPUT_DIR / f"segment-realmlp-td-revision-{REVISION}-{score_tag}-summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print(f"Saved Test predictions: {test_path}")